# 5 — Static INT8 quantization

**Runs in: Jupyter container on node-serve-model**

Static quantization requires a small calibration dataset so that the
quantizer can determine the optimal scale/zero-point for each activation.
This usually gives better accuracy than dynamic quantization.

We test two tolerance settings:

| Variant | `tolerable_loss` | Meaning |
|---|---|---|
| **Aggressive** | 0.05 | up to 5 % accuracy drop allowed |
| **Conservative** | 0.01 | up to 1 % accuracy drop allowed |

We generate synthetic calibration samples (random 64-dim vectors).
In production, use a representative sample from your listening-history dataset.

In [ ]:
# runs in Jupyter container on node-serve-model
import time
import numpy as np
import onnxruntime as ort
from neural_compressor import PostTrainingQuantConfig, quantization
from neural_compressor.config import AccuracyCriterion

ONNX_PATH          = "model_artifacts/smartqueue_ranker.onnx"
STATIC_AGG_PATH    = "model_artifacts/smartqueue_ranker_static_agg.onnx"
STATIC_CONS_PATH   = "model_artifacts/smartqueue_ranker_static_cons.onnx"

def benchmark_session(ort_session, input_dim=64, num_trials=500, batch_size=32):
    input_name = ort_session.get_inputs()[0].name
    single_sample = np.random.rand(1, input_dim).astype(np.float32)
    for _ in range(20):
        ort_session.run(None, {input_name: single_sample})
    latencies = []
    for _ in range(num_trials):
        t0 = time.time()
        ort_session.run(None, {input_name: single_sample})
        latencies.append(time.time() - t0)
    print(f"Inference Latency (single sample, p50): {np.percentile(latencies, 50)*1000:.2f} ms")
    print(f"Inference Latency (single sample, p95): {np.percentile(latencies, 95)*1000:.2f} ms")
    print(f"Inference Latency (single sample, p99): {np.percentile(latencies, 99)*1000:.2f} ms")
    batch = np.random.rand(batch_size, input_dim).astype(np.float32)
    batch_times = []
    for _ in range(200):
        t0 = time.time()
        ort_session.run(None, {input_name: batch})
        batch_times.append(time.time() - t0)
    print(f"Batch Throughput  (batch={batch_size}): {batch_size*200/sum(batch_times):.1f} samples/sec")

In [ ]:
# runs in Jupyter container on node-serve-model
# Build a calibration DataLoader from synthetic samples
# In production: replace with real (user_feat | song_feat) samples
from torch.utils.data import DataLoader, TensorDataset
import torch

calib_data = torch.randn(128, 64).numpy().astype(np.float32)
calib_dataset = TensorDataset(torch.tensor(calib_data))
calib_loader  = DataLoader(calib_dataset, batch_size=16)

In [ ]:
# runs in Jupyter container on node-serve-model
# Aggressive static quantization — tolerable accuracy loss: 5 %
config_agg = PostTrainingQuantConfig(
    approach="static",
    accuracy_criterion=AccuracyCriterion(tolerable_loss=0.05),
)
q_model_agg = quantization.fit(ONNX_PATH, config_agg, calib_dataloader=calib_loader)
q_model_agg.save(STATIC_AGG_PATH)
print(f"Saved aggressive static-quantized model → {STATIC_AGG_PATH}")

In [ ]:
# runs in Jupyter container on node-serve-model
# Conservative static quantization — tolerable accuracy loss: 1 %
config_cons = PostTrainingQuantConfig(
    approach="static",
    accuracy_criterion=AccuracyCriterion(tolerable_loss=0.01),
)
q_model_cons = quantization.fit(ONNX_PATH, config_cons, calib_dataloader=calib_loader)
q_model_cons.save(STATIC_CONS_PATH)
print(f"Saved conservative static-quantized model → {STATIC_CONS_PATH}")

In [ ]:
# runs in Jupyter container on node-serve-model
session_base = ort.InferenceSession(ONNX_PATH,       providers=["CPUExecutionProvider"])
session_agg  = ort.InferenceSession(STATIC_AGG_PATH, providers=["CPUExecutionProvider"])
session_cons = ort.InferenceSession(STATIC_CONS_PATH, providers=["CPUExecutionProvider"])

print("=== Baseline ONNX ===")
benchmark_session(session_base)
print()
print("=== Static INT8 (aggressive, tol=0.05) ===")
benchmark_session(session_agg)
print()
print("=== Static INT8 (conservative, tol=0.01) ===")
benchmark_session(session_cons)